# gRPC: When and Why

This notebook covers:

1. What gRPC actually is — Protobuf over HTTP/2 — and how that differs from REST-over-HTTP/1.1
2. Defining a service in a `.proto` file using the asset domain we've used throughout the curriculum
3. Generating Python stubs with `grpc_tools.protoc` and reading what they actually contain
4. Standing up an in-process gRPC server and calling it with a client, all inside one notebook
5. A server-streaming RPC for live prices — the use case where gRPC's HTTP/2 substrate genuinely outclasses REST
6. A direct comparison table: when gRPC wins, when REST wins, when the choice is a wash

**Scope**: `grpcio` + `grpcio-tools`. The gRPC server runs on a real `127.0.0.1:<free-port>` socket inside this notebook process so the round trips are honest HTTP/2 — not a mocked-in-memory shim. Every server is shut down at the end of its cell.

## 1. What gRPC Actually Is

gRPC is a **typed RPC framework**: you describe a service in a schema language (Protocol Buffers), generate stubs in your language of choice, and call methods that look like local function calls but are HTTP/2 requests carrying length-prefixed Protobuf messages.

The substrate matters. REST-over-HTTP/1.1 (what every previous notebook in this curriculum has shipped) is:

- text JSON over a per-request TCP connection (or, with keep-alive, a one-at-a-time pipeline)
- semantically organized around resources and verbs (`GET /assets/AAPL`)
- consumable by anything that speaks HTTP and JSON — every language, every browser, every curl

gRPC-over-HTTP/2 is:

- binary Protobuf over a long-lived multiplexed connection (many concurrent streams on one TCP socket)
- semantically organized around services and methods (`AssetService.GetAsset(ticker="AAPL")`)
- typed end-to-end via codegen — caller and callee compile against the same `.proto`
- four call patterns: unary, server-streaming, client-streaming, bidirectional streaming

The trade is wire compactness + type safety + streaming for a heavier toolchain and worse browser compatibility (browsers can't speak raw HTTP/2 from JS; you need a `grpc-web` proxy).

The rest of this notebook walks through a concrete `AssetService` so you've seen the shape end-to-end.

## 2. A `.proto` Definition

The `.proto` file is the contract. Both client and server are generated *from it* — drift is impossible by construction (the build fails if the proto and the code disagree).

Below is `assets.proto` for our usual domain. Read the `service` block at the bottom: each method names its request and response types, and `stream` is what makes a method server-streaming.

In [1]:
import shutil
import sys
import socket
import time
from pathlib import Path
from tempfile import mkdtemp

WORK = Path(mkdtemp(prefix="grpc_ch8_"))
print("workspace:", WORK)

PROTO = """\
syntax = "proto3";
package portfolio;

// One asset in the portfolio domain — same fields as the Pydantic Asset
// we've used since chapter 1. proto3 makes every field optional on the wire
// but the codegen still gives you typed accessors.
message Asset {
  string ticker = 1;   // field numbers are part of the wire format; never change them
  string name   = 2;
  double price  = 3;
}

message GetAssetRequest {
  string ticker = 1;
}

message PriceTick {
  string ticker  = 1;
  double price   = 2;
  int64  ts_ms   = 3;   // unix ms
}

// The service. Each `rpc` is a method on the generated stub.
service AssetService {
  // Unary RPC: one request, one response. The bread-and-butter shape.
  rpc GetAsset (GetAssetRequest) returns (Asset);

  // Server-streaming RPC: one request, many responses over one HTTP/2 stream.
  // Section 5 demonstrates this for live price ticks.
  rpc StreamPrices (GetAssetRequest) returns (stream PriceTick);
}
"""

(WORK / "assets.proto").write_text(PROTO, encoding="utf-8")
print("--- assets.proto ---")
print(PROTO)


workspace: C:\Users\mathi\AppData\Local\Temp\grpc_ch8_5xs8edso
--- assets.proto ---
syntax = "proto3";
package portfolio;

// One asset in the portfolio domain — same fields as the Pydantic Asset
// we've used since chapter 1. proto3 makes every field optional on the wire
// but the codegen still gives you typed accessors.
message Asset {
  string ticker = 1;   // field numbers are part of the wire format; never change them
  string name   = 2;
  double price  = 3;
}

message GetAssetRequest {
  string ticker = 1;
}

message PriceTick {
  string ticker  = 1;
  double price   = 2;
  int64  ts_ms   = 3;   // unix ms
}

// The service. Each `rpc` is a method on the generated stub.
service AssetService {
  // Unary RPC: one request, one response. The bread-and-butter shape.
  rpc GetAsset (GetAssetRequest) returns (Asset);

  // Server-streaming RPC: one request, many responses over one HTTP/2 stream.
  // Section 5 demonstrates this for live price ticks.
  rpc StreamPrices (GetAssetReques

Two `.proto` gotchas worth knowing before you write a real one:

- **Field numbers are forever.** The numbers (`= 1`, `= 2`, ...) are what's actually on the wire — the field *name* is purely a label for humans. Renaming a field is free; renumbering or reusing a number is a breaking wire-format change. Treat retired field numbers as poison: reserve them with `reserved 4, 5;` to prevent reuse.
- **proto3 made every field optional by default.** In proto2 you wrote `required` and `optional`; proto3 removed `required` entirely (it turned out to be a backwards-compat nightmare) and made everything optional. Use the explicit `optional` keyword if you need to distinguish "field absent" from "field set to its zero value".

## 3. Generating Python Stubs

`grpc_tools.protoc` is the codegen entrypoint. Give it the proto file and an output directory; it writes two modules:

- `assets_pb2.py` — the message classes (`Asset`, `GetAssetRequest`, `PriceTick`). Pure Protobuf, no networking.
- `assets_pb2_grpc.py` — the service classes: the `AssetServiceStub` (client side) and `AssetServiceServicer` (server side, abstract — you subclass it).

You can call protoc as a subprocess (`python -m grpc_tools.protoc ...`) or as a library, which is what the cell below does. The library form composes better in build scripts.

In [2]:
from grpc_tools import protoc

# protoc.main mirrors the CLI: argv-style list, returns 0 on success.
rc = protoc.main([
    "protoc",
    f"--proto_path={WORK}",
    f"--python_out={WORK}",
    f"--grpc_python_out={WORK}",
    str(WORK / "assets.proto"),
])
assert rc == 0, f"protoc failed with rc={rc}"

print("generated files:")
for f in sorted(WORK.glob("*.py")):
    print(f"  {f.name:<25} {f.stat().st_size:>5} bytes")

# Put WORK on sys.path so we can import the generated modules.
sys.path.insert(0, str(WORK))

# The generated _pb2 module imports each other via package-relative imports
# starting in modern grpcio-tools (>=1.59). Make sure no stale earlier copy
# is cached from a previous notebook run.
for stale in ("assets_pb2", "assets_pb2_grpc"):
    sys.modules.pop(stale, None)

import assets_pb2
import assets_pb2_grpc

# Inspect what we got: classes, not magic. Each message is a real Python class
# with attribute accessors that wrap the underlying Protobuf representation.
print("\nassets_pb2 exports        :", [n for n in dir(assets_pb2) if not n.startswith("_")][:8], "...")
print("assets_pb2_grpc exports   :", [n for n in dir(assets_pb2_grpc) if not n.startswith("_")][:6])
print("\nAsset is a class         :", assets_pb2.Asset)
print("AssetService stub class  :", assets_pb2_grpc.AssetServiceStub)


generated files:
  assets_pb2.py              2002 bytes
  assets_pb2_grpc.py         5148 bytes

assets_pb2 exports        : ['Asset', 'DESCRIPTOR', 'GetAssetRequest', 'PriceTick'] ...
assets_pb2_grpc exports   : ['AssetService', 'AssetServiceServicer', 'AssetServiceStub', 'GRPC_GENERATED_VERSION', 'GRPC_VERSION', 'add_AssetServiceServicer_to_server']

Asset is a class         : <class 'assets_pb2.Asset'>
AssetService stub class  : <class 'assets_pb2_grpc.AssetServiceStub'>


Two things to call out:

- **`assets_pb2.py` is generated.** It's not something you check in by hand; treat it as a build artifact. In a real project, codegen runs as part of `make` / `bazel` / `setup.py build`, and the output goes to a folder you `.gitignore`.
- **The stub vs the servicer.** `AssetServiceStub` is what the *client* instantiates against a channel and calls into. `AssetServiceServicer` is the *server* side — you subclass it and implement each method. The two are generated together from the same `.proto`; there is no way for them to disagree.

## 4. A Tiny Server and Client

A complete unary RPC round trip in three pieces:

1. Implement `AssetServiceServicer` with one method — `GetAsset`.
2. Start a `grpc.server` on a free port, register the servicer, call `start()` (returns immediately; the server runs on a thread pool).
3. Open a channel from the client, instantiate the stub, call `GetAsset(...)` — looks like a local call, is actually an HTTP/2 request.

In [3]:
import grpc
from concurrent import futures

# A simple in-memory backing store. Same data shape as previous notebooks.
PORTFOLIO = {
    "AAPL": ("Apple Inc.",   190.0),
    "MSFT": ("Microsoft",    420.0),
    "NVDA": ("NVIDIA Corp.", 900.0),
}

class AssetServiceImpl(assets_pb2_grpc.AssetServiceServicer):
    def GetAsset(self, request, context):
        ticker = request.ticker.upper()
        if ticker not in PORTFOLIO:
            # grpc statuses are richer than HTTP status codes. NOT_FOUND maps
            # to HTTP 404 in grpc-gateway, and to UNAVAILABLE-ish behaviour in
            # most clients. Set status + details and return an empty message.
            context.set_code(grpc.StatusCode.NOT_FOUND)
            context.set_details(f"unknown ticker: {ticker}")
            return assets_pb2.Asset()
        name, price = PORTFOLIO[ticker]
        return assets_pb2.Asset(ticker=ticker, name=name, price=price)

    def StreamPrices(self, request, context):
        # Implemented in section 5 — left empty here.
        context.set_code(grpc.StatusCode.UNIMPLEMENTED)
        return iter(())

def free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

# Start the server.
server = grpc.server(futures.ThreadPoolExecutor(max_workers=4))
assets_pb2_grpc.add_AssetServiceServicer_to_server(AssetServiceImpl(), server)
port = free_port()
server.add_insecure_port(f"127.0.0.1:{port}")
server.start()
print(f"grpc server up on 127.0.0.1:{port}")

# The client: open a channel, build a stub, call the method.
try:
    with grpc.insecure_channel(f"127.0.0.1:{port}") as channel:
        stub = assets_pb2_grpc.AssetServiceStub(channel)

        # Happy path.
        resp = stub.GetAsset(assets_pb2.GetAssetRequest(ticker="AAPL"))
        print(f"  GetAsset(AAPL) -> Asset(ticker={resp.ticker!r}, name={resp.name!r}, price={resp.price})")

        # Sad path: NOT_FOUND. The client side raises an RpcError.
        try:
            stub.GetAsset(assets_pb2.GetAssetRequest(ticker="BAD"))
        except grpc.RpcError as e:
            print(f"  GetAsset(BAD)  -> RpcError code={e.code().name}, details={e.details()!r}")
finally:
    server.stop(grace=1.0).wait()
    print("server stopped")


grpc server up on 127.0.0.1:61837
  GetAsset(AAPL) -> Asset(ticker='AAPL', name='Apple Inc.', price=190.0)
  GetAsset(BAD)  -> RpcError code=NOT_FOUND, details='unknown ticker: BAD'
server stopped


A few things worth internalizing from the cell above:

- **`server.start()` returns immediately.** The server runs requests on its own thread pool. To keep a real process alive you'd `server.wait_for_termination()`; in the notebook we just shut it down at the end of the cell.
- **Errors come back as `grpc.RpcError` with a status code and details string.** That's gRPC's equivalent of HTTP status codes — a fixed enum (`OK`, `NOT_FOUND`, `UNAUTHENTICATED`, `DEADLINE_EXCEEDED`, `INTERNAL`, ...). The mapping from your domain errors to gRPC codes is something you design once and apply consistently.
- **Type safety.** `assets_pb2.GetAssetRequest(ticker="AAPL")` is a typed call; if you typo the field name, you get a Python error before bytes hit the wire. Compare to `httpx.get("/assets/AAPL")` where the contract is implicit in the URL.
- **`insecure_channel` is for local development only.** In production gRPC runs over TLS via `grpc.secure_channel(target, credentials)`. The cert/key handling is the same kind of machinery you'd configure for any TLS endpoint.

## 5. Server-Streaming for Live Prices

This is the use case where gRPC's HTTP/2 substrate becomes a structural fit. A REST-equivalent ("poll `/assets/AAPL/price` every 200ms") wastes a round trip's worth of headers and TCP handshakes per tick. SSE (notebook 3.3) closes part of that gap but still rides on HTTP/1.1 framing.

gRPC's server-streaming RPC: one request, one stream of responses on **one HTTP/2 stream**. The client gets an iterator; each `next()` blocks until the server emits the next message. The connection survives across messages — no header re-send, no socket teardown.

Let's implement it. We'll send 5 simulated price ticks for AAPL over a single RPC call.

In [4]:
import random
import threading
import time as _time

class StreamingAssetServiceImpl(assets_pb2_grpc.AssetServiceServicer):
    def GetAsset(self, request, context):
        ticker = request.ticker.upper()
        if ticker not in PORTFOLIO:
            context.set_code(grpc.StatusCode.NOT_FOUND)
            return assets_pb2.Asset()
        name, price = PORTFOLIO[ticker]
        return assets_pb2.Asset(ticker=ticker, name=name, price=price)

    def StreamPrices(self, request, context):
        """Server-streaming RPC: yield as many PriceTicks as you want;
        the wire keeps one HTTP/2 stream open the whole time."""
        ticker = request.ticker.upper()
        if ticker not in PORTFOLIO:
            context.set_code(grpc.StatusCode.NOT_FOUND)
            context.set_details(f"unknown ticker: {ticker}")
            return
        _, base = PORTFOLIO[ticker]
        for _ in range(5):
            if not context.is_active():
                # The client cancelled or the deadline expired. Stop emitting.
                return
            price = base + random.uniform(-2.0, 2.0)
            yield assets_pb2.PriceTick(ticker=ticker, price=round(price, 2),
                                       ts_ms=int(_time.time() * 1000))
            _time.sleep(0.1)

# Start a fresh server for this section.
server = grpc.server(futures.ThreadPoolExecutor(max_workers=4))
assets_pb2_grpc.add_AssetServiceServicer_to_server(StreamingAssetServiceImpl(), server)
port = free_port()
server.add_insecure_port(f"127.0.0.1:{port}")
server.start()

t0 = _time.perf_counter()
try:
    with grpc.insecure_channel(f"127.0.0.1:{port}") as channel:
        stub = assets_pb2_grpc.AssetServiceStub(channel)

        # The stub method returns an iterator — each `for tick in ...`
        # consumes one server-emitted message off the same HTTP/2 stream.
        ticks = stub.StreamPrices(assets_pb2.GetAssetRequest(ticker="AAPL"))
        for tick in ticks:
            dt = (_time.perf_counter() - t0) * 1000
            print(f"  +{dt:>5.1f}ms  ticker={tick.ticker}  price={tick.price:>7.2f}  ts={tick.ts_ms}")
finally:
    server.stop(grace=1.0).wait()


  +  7.7ms  ticker=AAPL  price= 191.44  ts=1781507571784
  +107.7ms  ticker=AAPL  price= 189.55  ts=1781507571884
  +208.2ms  ticker=AAPL  price= 191.01  ts=1781507571985
  +308.8ms  ticker=AAPL  price= 190.59  ts=1781507572086
  +409.4ms  ticker=AAPL  price= 188.48  ts=1781507572186


Read the timestamps: about 100ms between ticks, all delivered on a single RPC. No new TCP connection, no re-handshake, no `GET /...` headers for each message. The client-side iterator is the entire API — the streaming machinery is transparent.

Two patterns to know about:

- **`context.is_active()`** is the server's way to detect cancellation. If the client closes the iterator early or the deadline passes, `is_active()` flips to `False` and you should stop emitting. Streaming without this check leaks work indefinitely.
- **Client-side deadlines.** `stub.StreamPrices(request, timeout=2.0)` would have ended the stream after 2 seconds even if the server kept emitting. Deadlines propagate across gRPC calls — if A calls B calls C with a 5-second deadline, C inherits the remaining time. This is one of the gRPC ergonomics that pays off in multi-service systems.

## 6. gRPC vs REST: When Each Wins

The honest comparison, dimension by dimension:

| Dimension              | gRPC (Protobuf + HTTP/2)                                  | REST (JSON + HTTP/1.1 or h2)                               |
| ---                    | ---                                                       | ---                                                        |
| **Wire size**          | Binary Protobuf — typically 2-10× smaller than JSON       | Text JSON — verbose but inspectable                        |
| **Latency**            | Lower (multiplexed HTTP/2, no per-call handshake)         | Higher per call; keep-alive helps but limited              |
| **Streaming**          | First-class: server, client, bidi                         | SSE (one direction), websockets (separate protocol)        |
| **Typing**             | Generated stubs — caller and callee can't drift           | OpenAPI is generated but consumers must opt in             |
| **Browser support**    | Needs `grpc-web` + a proxy (Envoy/grpc-gateway)           | Native — every browser, every fetch()                      |
| **Tooling discovery**  | `grpcurl`, `Postman` recently; less mature than curl/Postman for REST | `curl`, browser DevTools, every HTTP tool        |
| **Schema evolution**   | Field numbers + reserved are the contract; rules are explicit | Conventional (additive fields are safe; rename is risky) |
| **Auth / TLS**         | Mutual TLS is idiomatic; per-call metadata for tokens     | Same options, via headers                                  |
| **Observability**      | OpenTelemetry has full gRPC instrumentation               | Same — gRPC has slightly better baked-in tracing IDs       |
| **Cognitive load**     | Higher — protoc, codegen, generated code in your tree     | Lower — strings and JSON, debuggable in DevTools           |

**Pick gRPC when**:

- The traffic is **service-to-service inside your trust boundary**. No browsers in the path, internal latency matters, schema discipline pays for itself.
- You need **streaming as a first-class concept** — live data feeds, telemetry, long-running progress reports.
- The team is large enough that **typed contracts** prevent more breakage than the codegen burden adds.
- The dollar cost of bytes matters (mobile clients on metered connections, very high QPS), and Protobuf's compactness pays off.

**Pick REST when**:

- A **browser is in the path** without a proxy. `grpc-web` exists, but it adds an Envoy hop most teams don't want to operate.
- The API is **external** — third-party developers expect JSON and OpenAPI, not Protobuf and `.proto` files. The discoverability of `curl /docs` is a real product feature.
- The team is **small or polyglot**, and the cost of maintaining codegen pipelines in N languages outweighs the wire-size win.
- The endpoints are simple CRUD and you don't need streaming. REST + JSON + Pydantic is hard to beat for that shape — it's most of what every other notebook in this curriculum has shown.

The rule that holds in practice: **REST at the edge, gRPC between services**. Your public API is FastAPI + OpenAPI; the internal calls between your microservices use gRPC if you have several of them. Many real systems run both.

## Key Takeaways

- **gRPC is Protobuf-over-HTTP/2 with codegen.** The `.proto` is the contract; client and server are *generated from it* so drift is impossible.
- **Four call patterns**: unary, server-streaming, client-streaming, bidi-streaming. The streaming variants are gRPC's structural advantage over REST.
- **Field numbers are forever.** Treat retired numbers as poison; reserve them. Renaming a field is free; renumbering is a wire-breaking change.
- **`grpc.RpcError` with a status code** is gRPC's error envelope. The codes are a fixed enum — map your domain errors to them once, apply consistently.
- **Deadlines propagate across calls.** A 5s deadline at the edge becomes 2s by the time it reaches the third hop. This is one of gRPC's quiet ergonomic wins for multi-service systems.
- **REST at the edge, gRPC between services** is the rule that holds in practice. The capstone in this curriculum is REST-only — that's the right call for the surface a learner or a third party will interact with.
- **Capstone tie-in**: the capstone stays REST + FastAPI by design. The notes above are the path you'd walk if it grew into a multi-service system with internal price feeds, batch order matching, or any second service that the FastAPI app needs to call hot.

## Exercises

All exercises extend `assets.proto` and regenerate stubs with the protoc cell above.

**1. Add a `ListAssets` RPC.** It should be a unary RPC returning `repeated Asset`. Regenerate stubs, implement on the servicer, and call from a client. Then convert it to a server-streaming variant (`stream Asset`) so the client can start processing tickers before the server has emitted them all. Time both versions and report the difference for a 100-asset portfolio.

**2. Client-side timeouts and cancellation.** Add a `SlowGetAsset` RPC that sleeps for 5 seconds before returning. Call it with `stub.SlowGetAsset(req, timeout=1.0)` and confirm the `RpcError` has code `DEADLINE_EXCEEDED`. Then implement the *server-side* check via `context.is_active()` so the server stops doing work when the client cancels — print a "client cancelled" line from the servicer to prove it. The point: long RPCs that don't respect cancellation leak work.

**3. A gRPC-Gateway sketch.** Read about `grpc-gateway` (or `Envoy`'s gRPC-JSON transcoding) and write a short markdown cell describing how you'd put a JSON/REST shim in front of `AssetService` so a browser can call `GET /v1/assets/AAPL`. You don't have to install anything — the deliverable is the explanation of the moving pieces (proto annotations, gateway proxy, codegen target), and one paragraph on what behaviour you'd lose for the streaming RPCs (hint: server-streaming becomes SSE; bidi doesn't translate).

In [5]:
try:
    sys.path.remove(str(WORK))
except ValueError:
    pass
for mod in ("assets_pb2", "assets_pb2_grpc"):
    sys.modules.pop(mod, None)
print("workspace at end:", WORK)
print("(left in place for the exercises; remove with shutil.rmtree if you're done)")


workspace at end: C:\Users\mathi\AppData\Local\Temp\grpc_ch8_5xs8edso
(left in place for the exercises; remove with shutil.rmtree if you're done)
